# 2 &mdash; Pendulum swing-up: a non-autonomous system

Demonstration of **Section 4.2** (and the comparison of **Section 5.1**) of
[*B-LSTM-MIONet*](https://arxiv.org/abs/2311.16519), arXiv:2311.16519.

The damped pendulum

$$\ddot\theta\left(\tfrac14 m l^2 + I\right) + \tfrac12 m l g \sin\theta = u(t) - b\dot\theta$$

is **non-autonomous**: an exogenous torque $u(t)$ drives it. The LSTM branch therefore receives
the masked history of the *control*, $u_{(t_0,t_n]}$, instead of the state history &mdash; this is
the full MIONet of Eq. (4), with three inputs $\big(x(t_n),\, u_{(t_0,t_n]},\, h(t_n)\big)$.

Training controls are drawn from a Gaussian random field with RBF kernel length $l = 0.01$. Two
extra things are shown here that the Lorenz notebook does not need:

* an **out-of-distribution** test with the deterministic control $u(t) = \sin(t/2)$, which the
  operator never saw during training (paper Table 3);
* a comparison against the **local DeepONet** baseline of Section 5.1, which is given
  $u(t_n + h)$ directly instead of the history (paper Table 6).

In [ ]:
import logging
import os
import warnings
from pathlib import Path

# Locate the repository root, whether the kernel started in notebooks/ or at the root.
ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "pendulum.yaml").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("could not find the repository root (no configs/pendulum.yaml above cwd)")
    ROOT = ROOT.parent

# Paths inside a configuration file are relative to the working directory, so work from the
# repository root: data/, mlruns/ and figures/ are then created there (all three are git-ignored).
os.chdir(ROOT)
os.environ["MLFLOW_TRACKING_URI"] = str(ROOT / "mlruns")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # set before mlflow is imported
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

print("repository root     :", ROOT)
print("MLFLOW_TRACKING_URI :", os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch

from blstm_mionet.config import load_config
from blstm_mionet.data.datasets import prepare_torch_dataset, split_dataset
from blstm_mionet.data.generate import generate_trajectories, save_dataset
from blstm_mionet.evaluation.evaluate import evaluate_single_step
from blstm_mionet.models import build_model, dataset_preparer_for
from blstm_mionet.training import tracking
from blstm_mionet.training.trainer import train_adam
from blstm_mionet.utils.device import resolve_device
from blstm_mionet.utils.seed import set_seed

# blstm_mionet.evaluation.plotting selects the non-interactive "Agg" backend when it is
# imported, so ask for the inline backend again afterwards.
%matplotlib inline

print("torch               :", torch.__version__)

## Settings: `QUICK` versus the paper

`QUICK = True` keeps the notebook under about three minutes on a laptop CPU; drawing a Gaussian
random field per trajectory is the slowest part of the data generation. `QUICK = False` uses
`configs/pendulum.yaml` as shipped, which is the paper's setting.

| | `QUICK` | paper, Sec. 4.2 (`configs/pendulum.yaml`) |
| --- | --- | --- |
| trajectories / controls | 40 | 5000 states from $X_0 = \{\theta\in[-\pi,\pi],\ \dot\theta\in[-8,8]\}$ and 5000 GRF samples |
| horizon $T$ | 2 s | 10 s |
| RK-4 step $\Delta$ | 0.01 s | 0.01 s |
| masks per control | 40 | 10, giving $N_{\text{train}} = 50000$ |
| max spacing $h_{\max}$ | 0.02 s (`search_len = 2`) | 0.02 s |
| branch / trunk | width 100, depth 3 | width 200, depth 3 |
| LSTM | 64 units, 1 layer | 100 units, 2 layers |
| epochs | 5 | 1000, early stopping after 80 |
| test sub-sequences | 20 per trajectory | 200 per trajectory, 100 trajectories |

In [ ]:
QUICK = True

QUICK_OVERRIDES = [
    # data: 40 GRF-driven trajectories of 2 s instead of 5000 of 10 s
    "data.n_sample=40",
    "data.t_max=2.0",
    "data.output=data/pendulum_demo_train.npy",
    # a smaller operator
    "model.branch_state.width=100",
    "model.branch_memory.width=100",
    "model.branch_memory.lstm_size=64",
    "model.branch_memory.lstm_layer_num=1",
    "model.trunk.width=100",
    # training
    "training.datafile=data/pendulum_demo_train.npy",
    "training.search_num=40",
    "training.batch_size=64",
    "training.epochs=5",
    "training.device=cpu",
    "training.run_name=pendulum_quick",
    "training.registered_model_name=null",
    "training.plot_trajs=false",
    # inference
    "inference.search_num=20",
    "inference.device=cpu",
    "inference.plot_trajs=false",
]

# Paper scale: configs/pendulum.yaml already carries the Section 4.2 values.
FULL_OVERRIDES = [
    "training.device=0",
    "inference.device=0",
]

config = load_config(
    ROOT / "configs" / "pendulum.yaml", QUICK_OVERRIDES if QUICK else FULL_OVERRIDES
)

TRAIN_FILE = config.data.output
TEST_FILE = "data/pendulum_demo_test.npy" if QUICK else "data/pendulum_ctr_grf_N_100_h001_T10.npy"
OOD_FILE = "data/pendulum_demo_ood.npy" if QUICK else "data/pendulum_ctr_designate_N_100_h001_T10.npy"

print(f"architecture  : {config.model.architecture}")
print(f"control       : {config.data.control} (GRF kernel length a = {config.data.grf.a})")
print(f"trajectories  : {config.data.n_sample} x T = {config.data.t_max} s at dt = {config.data.step_size} s")
print(f"state         : component {config.training.state_component} = theta(t)")
print(f"epochs        : {config.training.epochs}")

## 1. Generate the trajectories

For a non-autonomous system `generate_trajectories` also returns `u` of shape
`[N_time, 1, N_sample]`. Three datasets are built: training and test controls sampled from the
Gaussian random field (`control: gaussian`), plus an out-of-distribution set driven by
$u(t) = \sin(t/2)$ (`control: designate`).

```bash
blstm-mionet generate --config configs/pendulum.yaml \
    --n-sample 40 --set data.t_max=2.0 --output data/pendulum_demo_train.npy

blstm-mionet generate --config configs/pendulum.yaml \
    --n-sample 10 --set data.t_max=2.0 --set data.control=designate \
    --output data/pendulum_demo_ood.npy
```

In [ ]:
def make_dataset(base, n_sample, seed, output, control=None):
    # Integrate ``n_sample`` trajectories with ``seed`` and save them to ``output``.
    cfg = copy.deepcopy(base)
    cfg.n_sample, cfg.seed, cfg.output = n_sample, seed, output
    if control is not None:
        cfg.control = control
    set_seed(cfg.seed)
    data = generate_trajectories(cfg)
    save_dataset(data, cfg.output)
    print(f"{output}: x {data['x'].shape}, u {data['u'].shape}, t {data['t'].shape}")
    return data


train_raw = make_dataset(config.data, config.data.n_sample, 999, TRAIN_FILE)
test_raw = make_dataset(config.data, 20 if QUICK else 100, 2024, TEST_FILE)
ood_raw = make_dataset(config.data, 20 if QUICK else 100, 2024, OOD_FILE, control="designate")

## 2. A look at the data and at the masking (paper Sec. 3.4)

The masking of Section 3.4 is applied to the **control**:

$$u_{(t_0,t_1]} = [u(t_0), u(t_1), 0, \dots, 0], \quad \dots, \quad
  u_{(t_0,t_n]} = [u(t_0), \dots, u(t_n)],$$

so one control function yields `search_num` training quartets
$\{x(t_n),\, u_{(t_0,t_n]},\, h(t_n),\, x(t_n+h(t_n))\}$ with histories of different lengths. The
trailing zeros are the padding mask consumed by `LSTM_MLP`. The cut $t_n$ is drawn uniformly from
$(0, T - h_{\max})$ and the spacing $h(t_n)$ from $[0, h_{\max}]$; the target state is interpolated
with a cubic spline.

This step lives inside `blstm-mionet train` / `infer`, so it has no CLI equivalent of its own.

In [ ]:
preparer = dataset_preparer_for(config.model.architecture)
print(f"{config.model.architecture} is prepared by {preparer.__name__}()\n")

train_split, _ = split_dataset(train_raw, test_size=0.0)
input_masked, x_n, x_next, t_params = preparer(
    data=train_split,
    state_component=config.training.state_component,
    search_len=config.training.search_len,
    search_num=config.training.search_num,
    search_random=config.training.search_random,
    offset=config.training.offset,
    t_max=config.training.t_max,
)

n_traj = train_split[1].shape[0]
t_grid = np.asarray(train_split[2]).squeeze()
dt = float(t_grid[1] - t_grid[0])
print(f"\n{input_masked.shape[0]} quartets = {config.training.search_num} masks x {n_traj} controls")
print(f"the masked input is u, not x: input {input_masked.shape}, x_n {x_n.shape}")

In [ ]:
SAMPLE = 0
picks = [s * n_traj + SAMPLE for s in range(3)]
colours = ["tab:blue", "tab:orange", "tab:green"]

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9.5, 6.4), sharex=True)
top.plot(t_grid, train_split[0][SAMPLE, :, 0], color="0.8", lw=1.0, label="full control $u(t)$")
for colour, k in zip(colours, picks, strict=True):
    t_n = t_params[k, 0]
    n_hist = int(round(t_n / dt)) + 1
    top.plot(t_grid[:n_hist], input_masked[k, :n_hist, 0], color=colour, lw=1.6,
             label=f"masked $u_{{(t_0,t_n]}}$, $t_n$ = {t_n:.2f} s")
top.set_ylabel("$u(t)$ (Nm)")
top.set_title("Masking of the control (paper Sec. 3.4) and the state it drives")
top.legend(fontsize=8)

bottom.plot(t_grid, train_split[1][SAMPLE, :, config.training.state_component],
            color="0.5", lw=1.0, label=r"$\theta(t)$")
for colour, k in zip(colours, picks, strict=True):
    bottom.plot(t_params[k, 0], x_n[k, 0], "o", color=colour, ms=7)
    bottom.plot(t_params[k, 2], x_next[k, 0], "*", color=colour, ms=13)
bottom.set_xlabel("$t$ (s)")
bottom.set_ylabel(r"$\theta(t)$ (rad)")
bottom.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"{'t_n (s)':>10} {'h (s)':>9} {'theta(t_n)':>12} {'theta(t_n+h)':>14} {'non-zero steps':>15}")
for k in picks:
    print(f"{t_params[k, 0]:10.3f} {t_params[k, 1]:9.4f} {x_n[k, 0]:12.3f} {x_next[k, 0]:14.3f} "
          f"{int(np.count_nonzero(input_masked[k])):15d}")

## 3. Build the operator

Same call as for Lorenz; the only difference is what the memory branch is fed. `build_model`
returns `LSTM_MIONet`: an MLP on $x(t_n)$, an MLP&rarr;LSTM&rarr;MLP on $u_{(t_0,t_n]}$, an MLP on
$h$, fused by a dot product plus the bias `tau` (paper Eq. 4).

In [ ]:
set_seed()
device = resolve_device(config.training.device, verbose=True)

prepare_kwargs = {
    "state_component": config.training.state_component,
    "search_len": config.training.search_len,
    "search_num": config.training.search_num,
    "search_random": config.training.search_random,
    "offset": config.training.offset,
    "t_max": config.training.t_max,
    "scale_mode": config.training.scale_mode,
    "device": device,
}

train_only, _ = split_dataset(train_raw, test_size=config.training.holdout_size)
train_dataset, _, state_feature_num = prepare_torch_dataset(train_only, preparer, **prepare_kwargs)

model = build_model(config.model, state_feature_num)
print(model)
print(f"\n{sum(p.numel() for p in model.parameters()):,} trainable parameters")

## 4. Train

```bash
blstm-mionet train --config configs/pendulum.yaml \
    --data data/pendulum_demo_train.npy --epochs 5 --device cpu --no-plot
```

In [ ]:
tracking.configure_tracking(config.tracking.uri)
with tracking.start_run(config.training.experiment_name, config.training.run_name) as run:
    RUN_ID = run.info.run_id
    tracking.log_config(config)
    history = train_adam(config.training, model, train_dataset, device)

MODEL_URI = history["model_uri"]
print(f"\nbest val_loss  : {history['best_metric']['val_loss']:.5f} (epoch {history['best_epoch']})")
print(f"MLflow run uri : runs:/{RUN_ID}")
print(f"model uri      : {MODEL_URI}")

## 5. Evaluate, in distribution and out of distribution

The same trained operator is evaluated twice: on held-out controls drawn from the *same* Gaussian
random field, and on the deterministic control $u(t) = \sin(t/2)$, which is not in the training
distribution at all. Table 3 of the paper reports 2.02 % and 2.88 % mean $L_2$ for these two cases.

Keep in mind that five epochs on forty trajectories is far from converged, and that the error is
*relative* per trajectory: a $\theta(t)$ that stays near zero divides by a small norm and blows the
mean up. The cell below therefore prints the median as well.

```bash
blstm-mionet infer --config configs/pendulum.yaml \
    --data data/pendulum_demo_test.npy --model runs:/<run id>/model \
    --set inference.search_num=20 --device cpu

blstm-mionet infer --config configs/pendulum.yaml \
    --data data/pendulum_demo_ood.npy --model runs:/<run id>/model \
    --set inference.search_num=20 --device cpu
```

In [ ]:
def evaluate_on(raw, model_to_test, infer_cfg, prepare_with, label):
    # Prepare ``raw`` as a pure test set and report the per-trajectory L2 error.
    print(f"\n########## {label} ##########")
    _, split = split_dataset(raw, test_size=1.0)
    dataset, _, _ = prepare_torch_dataset(
        split,
        prepare_with,
        state_component=infer_cfg.state_component,
        search_len=infer_cfg.search_len,
        search_num=infer_cfg.search_num,
        search_random=infer_cfg.search_random,
        offset=infer_cfg.offset,
        t_max=infer_cfg.t_max,
        scale_mode=infer_cfg.scale_mode,
        device=device,
    )
    return evaluate_single_step(infer_cfg, model_to_test, dataset)


inference = config.inference
reloaded = tracking.load_model(MODEL_URI, device)

in_dist = evaluate_on(test_raw, reloaded, inference, preparer, "in distribution: u ~ GRF")
out_dist = evaluate_on(ood_raw, reloaded, inference, preparer, "out of distribution: u = sin(t/2)")

In [ ]:
panels = [
    (in_dist, r"in distribution, $u\sim\mathcal{G}$"),
    (out_dist, r"out of distribution, $u=\sin(t/2)$"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (res, title) in zip(axes, panels, strict=True):
    order = np.argsort(res["t_next"][0])
    ax.plot(res["t_next"][0][order], res["y_true"][0][order], color="red", lw=1.4, label="True")
    ax.plot(res["t_next"][0][order], res["y_pred"][0][order], color="blue", lw=1.4, ls="--",
            label="LSTM-MIONet")
    ax.set_xlabel("$t$ (s)")
    ax.set_ylabel(r"$\theta(t)$ (rad)")
    ax.set_title(f"{title}\n$L_2$ = {res['L2_error_mat'][0] * 100:.2f} % on this trajectory")
    ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

print(f"{'dataset':<34}{'mean L2 %':>12}{'st.dev. L2 %':>14}{'median L2 %':>14}")
for name, res in [("in distribution (u ~ GRF)", in_dist), ("out of distribution (u = sin(t/2))", out_dist)]:
    errors = res["L2_error_mat"] * 100
    print(f"{name:<34}{errors.mean():12.2f}{errors.std():14.2f}{np.median(errors):14.2f}")
print("\nThe relative error is computed per trajectory, so a trajectory whose theta(t) happens to")
print("stay close to zero divides by a small norm and dominates the mean.  In QUICK mode, with")
print("five epochs and twenty test trajectories, the median is the more informative number.")

## 6. Comparison with the local DeepONet (paper Sec. 5.1)

The local DeepONet of Section 5.1 replaces the history of the control by a single value: it is
given $u(t_n + h)$, the control *at the prediction time*. In the package this is the
`DeepONet_Local` architecture, which the registry automatically pairs with
`prepare_future_local_predict_dataset` instead of `prepare_local_predict_dataset` &mdash; that is
the whole difference in the data pipeline, and `dataset_preparer_for` takes care of it.

On the pendulum the two are nearly tied (Table 6: 2.02 % versus 2.04 % mean $L_2$), which is the
point of the comparison: knowing the future control is about as informative as remembering the
past one. On the *autonomous* Lorenz system, where the local baseline has no control to look at,
the gap is large (Table 7: 1.29 % versus 5.23 %).

```bash
blstm-mionet train --config configs/pendulum.yaml \
    --set model.architecture=DeepONet_Local \
    --data data/pendulum_demo_train.npy --epochs 5 --device cpu \
    --run-name pendulum_quick_deeponet_local --no-plot
```

In [ ]:
local_config = copy.deepcopy(config)
local_config.model.architecture = "DeepONet_Local"
local_config.training.run_name = f"{config.training.run_name}_deeponet_local"

local_preparer = dataset_preparer_for(local_config.model.architecture)
print(f"{local_config.model.architecture} is prepared by {local_preparer.__name__}()\n")

set_seed()
local_train_dataset, _, local_features = prepare_torch_dataset(
    train_only, local_preparer, **prepare_kwargs
)
local_model = build_model(local_config.model, local_features)

with tracking.start_run(local_config.training.experiment_name, local_config.training.run_name):
    local_history = train_adam(local_config.training, local_model, local_train_dataset, device)

local_in_dist = evaluate_on(
    test_raw, local_model, local_config.inference, local_preparer, "DeepONet_Local, u ~ GRF"
)

In [ ]:
print(f"{'':<18}{'mean L2 %':>12}{'median L2 %':>14}{'paper Table 6 (mean)':>24}")
for name, res, paper in [("LSTM-MIONet", in_dist, "2.02"), ("DeepONet_Local", local_in_dist, "2.04")]:
    errors = res["L2_error_mat"] * 100
    print(f"{name:<18}{errors.mean():12.2f}{np.median(errors):14.2f}{paper:>24}")
print("\n(QUICK numbers come from 5 epochs on 40 trajectories and are not comparable in "
      "magnitude\n with the paper; only the ranking is meaningful, and even that is noisy here.)")

## Reproducing the paper

```bash
# 5000 GRF controls, T = 10 s, Delta = 0.01 s  ->  N_train = 50000 quartets
blstm-mionet generate --config configs/pendulum.yaml

# 100 in-distribution test trajectories and 100 out-of-distribution ones
blstm-mionet generate --config configs/pendulum.yaml --n-sample 100 \
    --output data/pendulum_ctr_grf_N_100_h001_T10.npy
blstm-mionet generate --config configs/pendulum.yaml --n-sample 100 \
    --set data.control=designate --output data/pendulum_ctr_designate_N_100_h001_T10.npy

# 1000 epochs of Adam with early stopping; registers models:/pendulum/latest
blstm-mionet train --config configs/pendulum.yaml --device 0

# Table 3, both columns
blstm-mionet infer --config configs/pendulum.yaml --model models:/pendulum/latest
blstm-mionet infer --config configs/pendulum.yaml --model models:/pendulum/latest \
    --data data/pendulum_ctr_designate_N_100_h001_T10.npy

# Table 6: the local DeepONet baseline
blstm-mionet train --config configs/pendulum.yaml --device 0 \
    --set model.architecture=DeepONet_Local --run-name deeponet_local \
    --set training.registered_model_name=pendulum_local
blstm-mionet infer --config configs/pendulum.yaml \
    --set model.architecture=DeepONet_Local --model models:/pendulum_local/latest
```

Published results for the pendulum, state $\theta(t)$:

| | in distribution | out of distribution |
| --- | --- | --- |
| mean $L_2$ (Table 3) | 2.02 % | 2.88 % |
| st. dev. $L_2$ (Table 3) | 1.46 % | 1.28 % |
| local DeepONet, mean / st. dev. $L_2$ (Table 6) | 2.04 % / 1.50 % | &mdash; |
| reSGLD ensemble, $M = 300$: $L_2$ / PICP (Table 4) | 3.34 % / 100.00 % | &mdash; |

The uncertainty row is the subject of [`04_uncertainty.ipynb`](04_uncertainty.ipynb).

All runs are recorded under `mlruns/`; browse them with

```bash
MLFLOW_ALLOW_FILE_STORE=true uv run mlflow ui --backend-store-uri mlruns
```

and open <http://127.0.0.1:5000>.